# 2. Anatomy of the Drifting Field

Notebook 1 used $\mathbf{V}$ as a black box. Here we take it apart. By the end you should be able to look at a
drifting run that is misbehaving and say *which part of the field is wrong*.

We cover:

1. what $\mathbf{V}$ looks like as a picture, and its two halves
2. the temperature $\tau$ — the single most consequential hyperparameter
3. the affinity matrix and the Sinkhorn-style double normalisation (paper Table 11)
4. positives vs negatives: how many of each (paper Table 2)
5. self-exclusion — why the `eye(N) * 1e6` line matters
6. Laplace vs Gaussian, and the conservativeness question
7. the identifiability gap: can $\mathbf{V}\approx\mathbf{0}$ with $q\neq p$?

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110

def kernel_logits(x, y, tau, kind="lap"):
    d = torch.cdist(x, y)
    return -d / tau if kind == "lap" else -(d ** 2) / (2 * tau ** 2)

def compute_V(x, y_pos, y_neg=None, tau=0.2, normalize="both", kind="lap",
              exclude_self=True, w_neg=None, return_parts=False):
    """General drifting field. Setting return_parts=True gives (V, V_plus, V_minus)."""
    N = x.shape[0]
    self_neg = y_neg is None
    if self_neg:
        y_neg = x
    lp = kernel_logits(x, y_pos, tau, kind)
    ln = kernel_logits(x, y_neg, tau, kind)
    if self_neg and exclude_self:
        ln = ln - torch.eye(N, device=x.device) * 1e6
    if w_neg is not None:
        ln = ln + math.log(w_neg)

    logit = torch.cat([lp, ln], dim=1)
    if normalize == "both":
        A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    elif normalize == "row":
        A = logit.softmax(-1)
    else:
        A = logit.exp() / logit.exp().sum()          # global scale only, keeps magnitudes sane
    P = y_pos.shape[0]
    A_pos, A_neg = A[:, :P], A[:, P:]
    W_pos = A_pos * A_neg.sum(1, keepdim=True)
    W_neg = A_neg * A_pos.sum(1, keepdim=True)
    Vp, Vn = W_pos @ y_pos, W_neg @ y_neg
    return (Vp - Vn, Vp, Vn) if return_parts else Vp - Vn

def mean_shift(x, S, tau, kind="lap", exclude_self=False):
    lg = kernel_logits(x, S, tau, kind)
    if exclude_self:
        lg = lg - torch.eye(x.shape[0], device=x.device) * 1e6
    return lg.softmax(-1) @ S - x

def bimodal(n, sep=2.0, sd=0.4):
    idx = torch.randint(0, 2, (n,))
    c = torch.tensor([[-sep, 0.0], [sep, 0.0]])
    return c[idx] + sd * torch.randn(n, 2)

grid = torch.linspace(-4.2, 4.2, 19)
GX, GY = torch.meshgrid(grid, grid, indexing="xy")
Q = torch.stack([GX.reshape(-1), GY.reshape(-1)], -1)

## 1. The field and its two halves

Using the explicit mean-shift form, $\mathbf{V}=\mathbf{V}^{+}_{p}-\mathbf{V}^{-}_{q}$. Let's draw all three for a
$q$ that is misplaced.

In [ ]:
p = bimodal(400)
q = 0.55 * torch.randn(400, 2) + torch.tensor([0.0, 1.6])       # sitting above the data

Vp = mean_shift(Q, p, tau=0.9)
Vn = mean_shift(Q, q, tau=0.9)

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for a, (V, title, col) in zip(ax, [(Vp, r"attraction  $V^+_p$", "tab:blue"),
                                   (Vn, r"repulsion source  $V^-_q$", "tab:orange"),
                                   (Vp - Vn, r"drift  $V = V^+_p - V^-_q$", "k")]):
    a.scatter(p[:, 0], p[:, 1], s=5, c="tab:blue", alpha=0.35)
    a.scatter(q[:, 0], q[:, 1], s=5, c="tab:orange", alpha=0.35)
    a.quiver(Q[:, 0], Q[:, 1], V[:, 0], V[:, 1], color=col, width=0.005, angles="xy")
    a.set_title(title); a.set_aspect("equal")
plt.tight_layout(); plt.show()

Middle panel: $\mathbf{V}^{-}_{q}$ points *toward* $q$ — it is the mean-shift field of the generated cloud. We
**subtract** it, so the effective force points away from wherever the model already has mass. The right panel shows the
net effect: pull down into the two data modes, push out of the orange blob.

The key structural point: repulsion is computed from a distribution that *changes as the model learns*. That is what
makes the equilibrium possible, and it is what a plain regression loss can never have.

## 2. Temperature $\tau$

$\tau$ sets what counts as "nearby". Its effect is not a smooth knob — it changes the *character* of the field.

In [ ]:
q_mid = 0.35 * torch.randn(400, 2)              # collapsed between the modes
taus = [0.15, 0.4, 1.0, 3.0]

fig, ax = plt.subplots(1, len(taus), figsize=(3.4 * len(taus), 3.6))
for a, t in zip(ax, taus):
    V = compute_V(Q, p, q_mid, tau=t, normalize="row")
    a.scatter(p[:, 0], p[:, 1], s=4, c="tab:blue", alpha=0.3)
    a.scatter(q_mid[:, 0], q_mid[:, 1], s=4, c="tab:orange", alpha=0.3)
    a.quiver(Q[:, 0], Q[:, 1], V[:, 0], V[:, 1], color="k", width=0.005, angles="xy")
    a.set_title(f"tau = {t}\nmean |V| = {V.norm(dim=-1).mean():.3f}", fontsize=9)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

* **$\tau$ too small** — the field is essentially "go to your nearest neighbour". Sharp, but almost no signal reaches
  points that are far from any sample: notice the arrows go dead in the corners. This is the flat-kernel regime.
* **$\tau$ well matched** — a coherent field over the whole region of interest.
* **$\tau$ too large** — everything is dragged toward the global mean, i.e. *into the empty valley between the modes*.
  The field is strong and wrong.

There is no single right answer, which is why the paper sums losses over $\tau\in\{0.02,0.05,0.2\}$ (after normalisation)
instead of tuning one. Let's quantify the effect on a real training run.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def spiral(n, device=device):
    t = torch.sqrt(torch.rand(n, device=device)) * 3.5 * math.pi
    r = 0.35 * t
    xy = torch.stack([r * torch.cos(t), r * torch.sin(t)], -1) + 0.12 * torch.randn(n, 2, device=device)
    return xy / 3.9

class Gen(nn.Module):
    def __init__(self, h=256, nl=4):
        super().__init__()
        L, d = [], 2
        for _ in range(nl):
            L += [nn.Linear(d, h), nn.SiLU()]; d = h
        L += [nn.Linear(d, 2)]
        self.net = nn.Sequential(*L)
    def forward(self, z): return self.net(z)

def tv2d(A, B, bins=24, lim=1.3):
    def h(z):
        H = torch.histogramdd(z.cpu(), bins=[bins, bins], range=[-lim, lim, -lim, lim])[0]
        return H / H.sum()
    return 0.5 * (h(A) - h(B)).abs().sum().item()

def norm_drift(V):
    return V / V.pow(2).mean().sqrt().clamp_min(1e-8)

def train(taus=(0.05, 0.2, 0.8), eta=0.05, normalize="both", N_neg=256, N_pos=256,
          drift_norm=True, exclude_self=True, kind="lap", steps=3000, seed=0):
    torch.manual_seed(seed)
    g = Gen().to(device); opt = torch.optim.Adam(g.parameters(), 2e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    raw = []
    for _ in range(steps):
        x = g(torch.randn(N_neg, 2, device=device))
        y_pos = spiral(N_pos)
        with torch.no_grad():
            Vs = [compute_V(x, y_pos, None, tau=t, normalize=normalize, kind=kind,
                            exclude_self=exclude_self) for t in taus]
            raw.append(sum(v.pow(2).mean().item() for v in Vs))
            V = eta * sum(norm_drift(v) if drift_norm else v for v in Vs)
        loss = F.mse_loss(x, (x + V).detach())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step(); sched.step()
    with torch.no_grad():
        gen = g(torch.randn(20000, 2, device=device))
    return gen, raw

ref = spiral(20000)
floor = tv2d(spiral(20000), ref)
print(f"finite-sample TV floor: {floor:.3f}\n")
print(f"{'temperatures':28s} {'TV':>7s}")
for taus in [(0.05,), (0.2,), (0.8,), (2.0,), (0.05, 0.2, 0.8)]:
    gen, _ = train(taus=taus)
    print(f"{str(taus):28s} {tv2d(gen, ref):7.3f}")

Read this table carefully, because it does **not** simply confirm the paper.

In this 2-D toy the single finest temperature ($\tau=0.05$, TV 0.069) beats the multi-temperature set (0.117). But look
at the spread across single values: 0.069, 0.221, 0.456, 0.619. **Choosing $\tau$ badly is catastrophic; the
multi-$\tau$ sum is never catastrophic.** In 2-D you can afford to sweep $\tau$ and take the best. In a 512-dimensional
feature space, across a dozen features from four encoder scales, with a training run that costs GPU-days, you cannot —
and that is the regime the paper's ablation (FID 10.62 / 8.67 / 8.96 single vs **8.46** for the set) lives in.

The honest statement: **the multi-temperature sum buys robustness, and at scale robustness beats a tuned single value.**
That is a different claim from "it is always better", and it is worth keeping straight.

## 3. The affinity matrix, and normalising along both axes

`compute_V` builds one matrix of logits, $[\,{-d^{+}}/\tau \;\|\; {-d^{-}}/\tau\,]$, of shape
$N\times(P+M)$, then normalises it. Three choices:

| variant | operation | paper's FID |
|---|---|---|
| `none` | $A=\exp(\text{logit})$ | 10.54 |
| `row` | softmax over $\mathbf{y}$ (the mean-shift normalisation) | 8.92 |
| `both` | $A=\sqrt{\text{softmax}_{\mathbf{y}}\cdot\text{softmax}_{\mathbf{x}}}$ | **8.46** |

The `both` variant is one Sinkhorn iteration in disguise: normalising rows and columns pushes $A$ toward a
doubly-stochastic matrix, i.e. toward a *transport plan* between the query set and the target set. Importantly, all
three satisfy $\mathbf{V}_{p,p}=\mathbf{0}$ — swapping the positive and negative column blocks is a permutation of
columns, which none of the three normalisations distinguishes. That is why even the worst variant merely degrades
rather than diverging.

In [ ]:
x_demo = torch.randn(24, 2) * 0.6
p_demo = bimodal(24, sd=0.3)
lp = kernel_logits(x_demo, p_demo, 0.6)
ln = kernel_logits(x_demo, x_demo, 0.6) - torch.eye(24) * 1e6
logit = torch.cat([lp, ln], 1)

mats = {"raw exp(logit)": logit.exp(),
        "softmax over y (row)": logit.softmax(-1),
        "softmax over x (col)": logit.softmax(-2),
        "geometric mean (both)": torch.sqrt(logit.softmax(-1) * logit.softmax(-2))}

fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
for a, (t, M) in zip(ax, mats.items()):
    im = a.imshow(M, aspect="auto", cmap="viridis")
    a.axvline(23.5, c="w", lw=1.5)
    a.set_title(f"{t}\nrow sums {M.sum(1).min():.2f}-{M.sum(1).max():.2f}", fontsize=8)
    a.set_xlabel("positives | negatives"); a.set_ylabel("query x")
    plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

In [ ]:
print(f"{'normalisation':16s} {'TV':>7s}")
for nm in ["none", "row", "both"]:
    gen, _ = train(normalize=nm)
    print(f"{nm:16s} {tv2d(gen, ref):7.3f}")

`none` is clearly worst, matching the paper. But `row` edges out `both` here (0.096 vs 0.117) — the reverse of the
paper's ImageNet ordering. The double normalisation is a *transport-plan* correction: it matters when many queries
compete for the same few nearby positives, which is a high-dimensional, large-batch phenomenon. With 256 points in 2-D
there is little competition to correct.

What is robust across all three variants and both settings: nothing diverges. That is the equilibrium property doing
the heavy lifting — all three normalisations satisfy $\mathbf{V}_{p,p}=\mathbf{0}$, so the worst case is degradation,
never collapse.

## 4. How many positives and negatives?

Notebook 0b showed the drift estimate converges at $N^{-1/2}$. The paper's Table 2 fixes the compute budget and
shows that trading class diversity for sample count in *either* the positive or the negative set improves FID:
$N_{\text{pos}}$ 1→64 takes FID from 20.4 to 8.5. Let's confirm the direction of the effect at fixed total work.

In [ ]:
print(f"{'N_neg':>6s} {'N_pos':>6s} {'TV':>7s}")
for N_neg, N_pos in [(64, 16), (64, 64), (64, 256), (256, 64), (256, 256), (512, 512)]:
    gen, _ = train(N_neg=N_neg, N_pos=N_pos)
    print(f"{N_neg:6d} {N_pos:6d} {tv2d(gen, ref):7.3f}")

More of either kind helps; the cost is quadratic in batch size for the distance matrix, but that matrix is cheap
compared with the network. Note the asymmetry: negatives come free (they are the samples you already generated),
whereas positives require a data loader — which is why the paper implements a **sample queue** (a MoCo-style
per-class FIFO of 128 real samples) rather than a bespoke loader. See Notebook 3.

## 5. Self-exclusion

`y_neg = x`, so the distance matrix has zeros on the diagonal: every sample is its own nearest negative, with maximal
kernel weight. Without masking, that self-term dominates the repulsion and injects a constant bias. The paper's
Algorithm 2 adds `dist_neg += eye(N) * 1e6`. What does dropping it cost?

In [ ]:
# 3 seeds, so we can tell a real effect from run-to-run noise
for N in [256, 32]:
    line = f"batch N_neg = {N:4d} : "
    for ex in [True, False]:
        vs = [tv2d(train(exclude_self=ex, N_neg=N, N_pos=N, seed=s)[0], ref) for s in range(3)]
        line += f"exclude_self={str(ex):5s} TV = {np.mean(vs):.3f} +- {np.std(vs):.3f}   "
    print(line)

**The honest reading: in this 2-D toy the effect is inside the seed-to-seed noise.** Masking neither helps nor hurts
measurably at either batch size. So why does Algorithm 2 do it?

The mechanism is real but small. The self term contributes
$\bar k(\mathbf{x},\mathbf{x})(\mathbf{x}-\mathbf{x})=\mathbf{0}$ to the displacement — but it *steals normalising
mass*, and it is the single largest kernel weight since its distance is exactly zero. It therefore shrinks the
repulsion by a factor $\approx(1-\bar k_{\text{self}})$. With $\tau$ small relative to typical inter-sample distances,
$\bar k_{\text{self}}$ is close to 1 and the bias is severe; with $\tau$ large it is $O(1/N)$ and negligible. Our toy
sits at the benign end.

Two reasons to always mask anyway: it costs one line, and it has a theoretical role.
[arXiv:2605.22795](https://arxiv.org/abs/2605.22795) shows the leave-one-out field is what removes a
"reciprocal-KDE self-interaction" term from the entropy identity — precisely the term that otherwise blocks a clean
finite-particle convergence rate.

This is also a useful lesson about toy experiments in general: **a 2-D reproduction confirms mechanisms, not
magnitudes.** Several effects in this notebook (double normalisation, Laplace vs Gaussian, self-exclusion) are
measurable at ImageNet scale and invisible here, because they are all corrections to *high-dimensional,
large-batch* behaviour.

## 6. Laplace vs Gaussian

The paper uses Laplace. Notebook 0a showed why the choice is structural: with a **Gaussian** kernel the mean shift is
exactly $\tau^2\nabla\log\rho_\tau$, so the field is a gradient (conservative) and drifting is literally a
Wasserstein gradient flow of $\mathrm{KL}$. With **Laplace** it is not. Empirically, Laplace wins.

In [ ]:
# NOTE: tau enters differently (d/tau vs d^2/2tau^2), so sweep both to compare fairly.
for kind in ["lap", "gauss"]:
    row = f"kernel={kind:6s} "
    for taus in [(0.05,), (0.2,), (0.05, 0.2, 0.8)]:
        gen, _ = train(kind=kind, taus=taus)
        row += f" taus={str(taus):18s} TV={tv2d(gen, ref):.3f}  "
    print(row)

In this 2-D toy the Gaussian is competitive or better — which is unsurprising, since here it *is* an exact
Wasserstein gradient flow with a Lyapunov function. The paper's preference for Laplace is a high-dimensional
statement, and the mechanism is the exponent: with $d\sim\sqrt{C}$ growing with dimension, $\exp(-d^2/2\tau^2)$
collapses to zero far faster than $\exp(-d/\tau)$. The Laplace kernel keeps a usable dynamic range where the Gaussian
is already numerically flat, and its cusp at the origin makes it much more discriminative between "on top of a real
sample" and "merely near one".

The literature is genuinely split, and the split is informative:

* [arXiv:2605.22795](https://arxiv.org/abs/2605.22795) introduces "sharp-normalised" fields that recover a gradient
  structure for radial kernels, and proves finite-particle rates for both the conservative and the non-conservative
  Laplace dynamics — the latter needing an extra, unavoidable *scale-mismatch* residual.
* the force-guided molecular work (Notebook 9) deliberately **switches back to Gaussian**, because the whole method
  rests on the score identity $\mathbf{V}^{+}=\tau^2\,\mathbb{E}[\bar k\,\nabla\log p]$, which only holds cleanly for
  the Gaussian.

So: **Laplace for high-dimensional perceptual data, Gaussian when you need the score identity.**

## 7. The identifiability gap

Anti-symmetry gives $q=p\Rightarrow\mathbf{V}=\mathbf{0}$. The converse is what we actually want and it is **not**
a theorem. The paper's Appendix C.1 gives a heuristic: expand $p$ and $q$ in a basis $\{\varphi_i\}$ with coefficients
$\mathbf{a},\mathbf{b}$; the kernel construction makes $\mathbf{V}$ evaluated at $N$ probe points a *bilinear* form
$\sum_{ij}a_ib_j\mathbf{U}_{ij}$ with $\mathbf{U}_{ij}=-\mathbf{U}_{ji}$. Zero drift forces $a_ib_j-a_jb_i=0$ for all
$i,j$, i.e. $\mathbf{a}\parallel\mathbf{b}$; since both integrate to 1, $\mathbf{a}=\mathbf{b}$. The assumption doing
the work is that the $\mathbf{U}_{ij}$ are linearly independent, which needs the probes to be diverse and
$dN \gg m^2$.

Where does that break? When the kernel cannot *see* a difference: if $\tau$ is so large that the field only depends on
the mean, then any $q$ with the right mean is an equilibrium. Let's exhibit it.

In [ ]:
p_test = bimodal(2000, sep=2.0, sd=0.3)
q_wrong = 1.0 * torch.randn(2000, 2)                 # same mean, wrong shape (unimodal)
probes = torch.randn(600, 2) * 2.0

print(f"{'tau':>6s} {'mean |V| (q wrong)':>20s} {'mean |V| (q = p)':>18s} {'ratio':>8s}")
for t in [0.2, 0.5, 1.0, 3.0, 10.0, 30.0]:
    V_wrong = compute_V(probes, p_test, q_wrong, tau=t, normalize="row")
    V_ok    = compute_V(probes, p_test, bimodal(2000, sep=2.0, sd=0.3), tau=t, normalize="row")
    a, b = V_wrong.norm(dim=-1).mean().item(), V_ok.norm(dim=-1).mean().item()
    print(f"{t:6.1f} {a:20.4f} {b:18.4f} {a/b:8.2f}")

At sensible $\tau$ the field is 3–10x larger for the wrong $q$ than for a correct one — it *discriminates*. As
$\tau\to\infty$ the ratio collapses toward 1: the field becomes blind to everything but the first moment, and a wrong
$q$ becomes a near-equilibrium. **Identifiability is a property of the kernel's resolution, not of the framework.**

Practical corollaries:

* Use several $\tau$, including small ones. A small $\tau$ is what certifies fine structure.
* Make sure the *metric* is meaningful. In pixel space, Euclidean distance between images is a poor notion of
  similarity, so no $\tau$ resolves semantics — hence the feature encoder (Notebook 4).
* Log the raw $\|\mathbf{V}\|^2$ *per temperature*. If the small-$\tau$ term is flat and only the large-$\tau$ term is
  active, you are matching moments, not distributions.

## Summary

| knob | what it controls | failure at each extreme |
|---|---|---|
| $\tau$ | resolution of "nearby" | too small: starved field; too large: moment matching only |
| multi-$\tau$ | covers scales without tuning | — |
| normalisation | how mass is shared across the batch | `none` degrades ~2 FID; all preserve equilibrium |
| $N_{\text{pos}}, N_{\text{neg}}$ | variance of the field estimate | small: noisy drift, unstable training |
| self-exclusion | leave-one-out correctness | without: repulsion systematically weakened |
| kernel | conservative vs not | Gaussian: exact gradient flow; Laplace: sharper, empirically better |

Next: [3. Practical Drifting](<3. Practical Drifting.ipynb>).